# AMI Knowledge Graph Construction

This notebook extracts AMI English site data, prepares staff records for LLM extraction, and writes the resulting graph to Neo4j.

## Goal
- extract `staff`, `labs`, and `news`
- flatten staff records for semantic extraction
- build `Person`, `Course`, `Lab`, and `NewsArticle` nodes
- connect labs to people and news to mentioned people
- validate the resulting graph


## Sources And Graph Scope

### Public sources
- Main AMI English site: https://ami.lnu.edu.ua/en/
- Staff page: https://ami.lnu.edu.ua/en/about/staff
- Labs page: https://ami.lnu.edu.ua/en/about/labs
- News pages: https://ami.lnu.edu.ua/en/news and page 2

### Graph schema used in this notebook
Nodes:
- `Faculty`
- `Department`
- `Person`
- `Course`
- `Lab`
- `NewsArticle`

Relationships:
- `(:Faculty)-[:HAS_DEPARTMENT]->(:Department)`
- `(:Department)-[:EMPLOYS]->(:Person)`
- `(:Person)-[:TEACHES]->(:Course)`
- `(:Faculty)-[:HAS_LAB]->(:Lab)`
- `(:Person)-[:LEADS]->(:Lab)`
- `(:NewsArticle)-[:MENTIONS]->(:Person)`


In [ ]:
%pip install -r requirements.txt


In [1]:
import json
import logging
import os
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from neo4j import GraphDatabase

from extract import extract_labs, extract_news, extract_staff
from graphrag_build_kg import (
    FACULTY_NAME,
    FACULTY_URL,
    build_client,
    build_text_chunks,
    configure_api_rate_limit,
    course_id,
    deduplicate_entries,
    extract_chunk,
    require_env,
)
from json_to_txt import RECORD_SEPARATOR, render_employee

load_dotenv()

STAFF_JSON_PATH = Path('staff_with_courses_en.json')
COMBINED_JSON_PATH = Path('ami_en_site_data.json')
STAFF_FLAT_PATH = Path('staff_with_courses_en_flat.txt')
MAX_TEXT_CHUNKS = int(os.getenv('MAX_TEXT_CHUNKS', '12'))
CHUNK_SIZE = int(os.getenv('UNSTRUCTURED_CHUNK_SIZE', '5000'))
CHUNK_OVERLAP = int(os.getenv('UNSTRUCTURED_CHUNK_OVERLAP', '500'))
CLEAR_SEMANTIC_GRAPH = os.getenv('CLEAR_SEMANTIC_GRAPH', 'false').lower() == 'true'
DATABASE = os.getenv('NEO4J_DATABASE')

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')
logger = logging.getLogger('ami_notebook_builder')


## Step 1. Extract Structured Site Data

This step uses the local scraper module and writes two files:
- `staff_with_courses_en.json` for the original staff pipeline
- `ami_en_site_data.json` for the combined staff, labs, and news payload


In [2]:
staff = extract_staff()
labs = extract_labs()
news = extract_news()

STAFF_JSON_PATH.write_text(json.dumps(staff, ensure_ascii=False, indent=2), encoding='utf-8')
COMBINED_JSON_PATH.write_text(
    json.dumps({'staff': staff, 'labs': labs, 'news': news}, ensure_ascii=False, indent=2),
    encoding='utf-8',
)

print(f'Saved {len(staff)} staff records to {STAFF_JSON_PATH}')
print(f'Saved {len(labs)} labs and {len(news)} news items to {COMBINED_JSON_PATH}')
staff[:2], labs[:2], news[:2]


Saved 153 staff records to staff_with_courses_en.json
Saved 6 labs and 8 news items to ami_en_site_data.json


([{'section': 'Деканат',
   'name': 'Ivan Dyyak',
   'title': 'Декан',
   'position': 'Декан',
   'profile_url': 'https://ami.lnu.edu.ua/en/employee/dyyak',
   'email': 'ivan.dyyak@lnu.edu.ua',
   'profile_slug': 'dyyak',
   'courses': [{'name': 'Numerical Methods of Mathematical Physics (System Analysis)',
     'url': 'https://ami.lnu.edu.ua/en/course/numerical-methods-of-mathematical-physics-system-analysis'}]},
  {'section': 'Деканат',
   'name': 'Vitaliy Horlatch',
   'title': 'Заступник декана',
   'position': 'Заступник декана',
   'profile_url': 'https://ami.lnu.edu.ua/en/employee/horlatch',
   'email': 'vitaliy.horlatch@lnu.edu.ua',
   'profile_slug': 'horlatch',
   'courses': [{'name': 'Electronic Information Organization and Processing (cs)',
     'url': 'https://ami.lnu.edu.ua/en/course/electronic-information-organisation-and-processing-informatics'}]}],
 [{'name': 'Educational Laboratory of Computer Modeling',
   'url': 'https://ami.lnu.edu.ua/en/laboratory/computer-modelin

## Step 2. Flatten Staff Records For LLM Extraction

Only staff records are flattened, because the LLM extraction phase is still used for department and course relationships. Labs and news are imported deterministically later in this notebook.


In [3]:
records = [render_employee(employee) for employee in staff]
flat_output = f'\n{RECORD_SEPARATOR}\n'.join(records) + f'\n{RECORD_SEPARATOR}\n'
STAFF_FLAT_PATH.write_text(flat_output, encoding='utf-8')
print(f'Saved {len(records)} flattened staff records to {STAFF_FLAT_PATH}')
print(STAFF_FLAT_PATH.read_text(encoding='utf-8')[:1200])


Saved 153 flattened staff records to staff_with_courses_en_flat.txt
section: Деканат
name: Ivan Dyyak
title: Декан
position: Декан
profile_url: https://ami.lnu.edu.ua/en/employee/dyyak
email: ivan.dyyak@lnu.edu.ua
profile_slug: dyyak
courses: ["Numerical Methods of Mathematical Physics (System Analysis) (https://ami.lnu.edu.ua/en/course/numerical-methods-of-mathematical-physics-system-analysis)"]
section: Деканат
name: Vitaliy Horlatch
title: Заступник декана
position: Заступник декана
profile_url: https://ami.lnu.edu.ua/en/employee/horlatch
email: vitaliy.horlatch@lnu.edu.ua
profile_slug: horlatch
courses: ["Electronic Information Organization and Processing (cs) (https://ami.lnu.edu.ua/en/course/electronic-information-organisation-and-processing-informatics)"]
section: Деканат
name: Uliana Khimka
title: Заступник декана
position: Заступник декана
profile_url: https://ami.lnu.edu.ua/en/employee/himka-u-t
email: ulyana.khimka@lnu.edu.ua
profile_slug: himka-u-t
courses: []
section: Дека

## Step 3. LLM Extraction For Staff And Courses

This step reuses the helper functions from `graphrag_build_kg.py` so the notebook and script stay aligned.


In [4]:
configure_api_rate_limit()
raw_text = STAFF_FLAT_PATH.read_text(encoding='utf-8')
chunks = build_text_chunks(
    text=raw_text,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP,
    limit=MAX_TEXT_CHUNKS,
)
print('Chunks:', len(chunks))
print(chunks[0][:1000] if chunks else 'No chunks generated')


Chunks: 12
section: Деканат
name: Ivan Dyyak
title: Декан
position: Декан
profile_url: https://ami.lnu.edu.ua/en/employee/dyyak
email: ivan.dyyak@lnu.edu.ua
profile_slug: dyyak
courses: ["Numerical Methods of Mathematical Physics (System Analysis) (https://ami.lnu.edu.ua/en/course/numerical-methods-of-mathematical-physics-system-analysis)"]


section: Деканат
name: Vitaliy Horlatch
title: Заступник декана
position: Заступник декана
profile_url: https://ami.lnu.edu.ua/en/employee/horlatch
email: vitaliy.horlatch@lnu.edu.ua
profile_slug: horlatch
courses: ["Electronic Information Organization and Processing (cs) (https://ami.lnu.edu.ua/en/course/electronic-information-organisation-and-processing-informatics)"]


section: Деканат
name: Uliana Khimka
title: Заступник декана
position: Заступник декана
profile_url: https://ami.lnu.edu.ua/en/employee/himka-u-t
email: ulyana.khimka@lnu.edu.ua
profile_slug: himka-u-t
courses: []


section: Деканат
name: Roman Seliverstov
title: Заступник декана

In [5]:
client = build_client()
semantic_entries = []

for index, chunk in enumerate(chunks, start=1):
    logger.info('Extracting semantic graph from chunk %s/%s', index, len(chunks))
    semantic_entries.extend(extract_chunk(client, chunk))

semantic_entries = deduplicate_entries(semantic_entries)
print('Extracted semantic entries:', len(semantic_entries))
semantic_entries[:2]


INFO ami_notebook_builder: Extracting semantic graph from chunk 1/12
INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO ami_notebook_builder: Extracting semantic graph from chunk 2/12
INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO ami_notebook_builder: Extracting semantic graph from chunk 3/12
INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO ami_notebook_builder: Extracting semantic graph from chunk 4/12
INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO ami_notebook_builder: Extracting semantic graph from chunk 5/12
INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO ami_notebook_builder: Extracting semantic graph from chunk 6/12
INFO httpx: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
INFO ami_notebook_builder: Extract

Extracted semantic entries: 97


[{'department': {'name': 'Faculty of Applied Mathematics and Informatics, Ivan Franko National University of Lviv'},
  'person': {'name': 'Ivan Dyyak',
   'title': 'Декан',
   'position': 'Декан',
   'email': 'ivan.dyyak@lnu.edu.ua',
   'profile_slug': 'dyyak',
   'profile_url': 'https://ami.lnu.edu.ua/en/employee/dyyak'},
  'courses': [{'name': 'Numerical Methods of Mathematical Physics (System Analysis)',
    'url': 'https://ami.lnu.edu.ua/en/course/numerical-methods-of-mathematical-physics-system-analysis'}]},
 {'department': {'name': 'Faculty of Applied Mathematics and Informatics, Ivan Franko National University of Lviv'},
  'person': {'name': 'Vitaliy Horlatch',
   'title': 'Заступник декана',
   'position': 'Заступник декана',
   'email': 'vitaliy.horlatch@lnu.edu.ua',
   'profile_slug': 'horlatch',
   'profile_url': 'https://ami.lnu.edu.ua/en/employee/horlatch'},
  'courses': [{'name': 'Electronic Information Organization and Processing (cs)',
    'url': 'https://ami.lnu.edu.ua

## Step 4. Build The Expanded Neo4j Graph

Staff and course relationships come from the LLM extraction output. Labs and news are imported directly from the structured scraper output.


In [6]:
def create_constraints(driver, database: str | None) -> None:
    statements = [
        "CREATE CONSTRAINT faculty_name IF NOT EXISTS FOR (f:Faculty) REQUIRE f.name IS UNIQUE",
        "CREATE CONSTRAINT department_name IF NOT EXISTS FOR (d:Department) REQUIRE d.name IS UNIQUE",
        "CREATE CONSTRAINT person_name IF NOT EXISTS FOR (p:Person) REQUIRE p.name IS UNIQUE",
        "CREATE CONSTRAINT course_id IF NOT EXISTS FOR (c:Course) REQUIRE c.course_id IS UNIQUE",
        "CREATE CONSTRAINT lab_name IF NOT EXISTS FOR (l:Lab) REQUIRE l.name IS UNIQUE",
        "CREATE CONSTRAINT news_url IF NOT EXISTS FOR (n:NewsArticle) REQUIRE n.url IS UNIQUE",
    ]
    
    with driver.session(database=database) as session:
        for statement in statements:
            session.run(statement)

def clear_graph(driver, database: str | None) -> None:
    query = '''
    MATCH (n)
    WHERE n:Faculty OR n:Department OR n:Person OR n:Course OR n:Lab OR n:NewsArticle
    DETACH DELETE n
    '''
    with driver.session(database=database) as session:
        session.run(query)

def merge_person(session, person_name: str | None) -> None:
    if not person_name:
        return
    session.run('MERGE (:Person {name: $name})', {'name': person_name})

def import_staff_semantic_entries(driver, database: str | None, entries: list[dict[str, Any]]) -> None:
    query = '''
    MERGE (f:Faculty {name: $faculty.name})
    SET f.url = $faculty.url

    MERGE (p:Person {name: $person.name})
    SET p.profile_slug = $person.profile_slug,
        p.title = $person.title,
        p.position = $person.position,
        p.email = $person.email,
        p.profile_url = $person.profile_url

    FOREACH (_ IN CASE WHEN $department IS NULL THEN [] ELSE [1] END |
        MERGE (d:Department {name: $department.name})
        MERGE (f)-[:HAS_DEPARTMENT]->(d)
        MERGE (d)-[:EMPLOYS]->(p)
    )

    WITH f, p, $courses AS courses
    UNWIND courses AS course
    MERGE (c:Course {course_id: course.course_id})
    SET c.name = course.name,
        c.url = course.url
    MERGE (p)-[:TEACHES]->(c)
    '''

    with driver.session(database=database) as session:
        for entry in entries:
            courses = [
                {
                    'course_id': course_id(course),
                    'name': course['name'],
                    'url': course.get('url'),
                }
                for course in entry['courses']
            ]
            payload = {
                'faculty': {'name': FACULTY_NAME, 'url': FACULTY_URL},
                'department': entry.get('department'),
                'person': entry['person'],
                'courses': courses,
            }
            session.run(query, payload)

def import_labs(driver, database: str | None, labs: list[dict[str, Any]]) -> None:
    query = '''
    MERGE (f:Faculty {name: $faculty.name})
    SET f.url = $faculty.url
    MERGE (l:Lab {name: $lab.name})
    SET l.url = $lab.url,
        l.email = $lab.email,
        l.phone = $lab.phone,
        l.phone_url = $lab.phone_url
    MERGE (f)-[:HAS_LAB]->(l)
    WITH l, $head_name AS head_name, $head_profile_url AS head_profile_url
    FOREACH (_ IN CASE WHEN head_name IS NULL OR head_name = '' THEN [] ELSE [1] END |
        MERGE (p:Person {name: head_name})
        SET p.profile_url = coalesce(p.profile_url, head_profile_url)
        MERGE (p)-[:LEADS]->(l)
    )
    '''
    with driver.session(database=database) as session:
        for lab in labs:
            session.run(
                query,
                {
                    'faculty': {'name': FACULTY_NAME, 'url': FACULTY_URL},
                    'lab': lab,
                    'head_name': lab.get('head_name'),
                    'head_profile_url': lab.get('head_profile_url'),
                },
            )

def normalize_text(value: str | None) -> str:
    if not value:
        return ''
    return ' '.join(value.lower().split())

def build_person_lookup(entries: list[dict[str, Any]], labs: list[dict[str, Any]]) -> list[dict[str, str]]:
    lookup: dict[str, dict[str, str]] = {}
    for entry in entries:
        person = entry.get('person', {})
        name = (person.get('name') or '').strip()
        if name:
            lookup[normalize_text(name)] = {'name': name}
    for lab in labs:
        head_name = (lab.get('head_name') or '').strip()
        if head_name:
            lookup.setdefault(normalize_text(head_name), {'name': head_name})
    return list(lookup.values())

def detect_mentions(article: dict[str, Any], person_lookup: list[dict[str, str]]) -> list[str]:
    haystack = normalize_text(' '.join([
        article.get('title') or '',
        article.get('preview') or '',
        article.get('content') or '',
    ]))
    matches = []
    for person in person_lookup:
        name = person['name']
        if normalize_text(name) and normalize_text(name) in haystack:
            matches.append(name)
    return sorted(set(matches))

def import_news(driver, database: str | None, news: list[dict[str, Any]], person_lookup: list[dict[str, str]]) -> None:
    article_query = '''
    MERGE (n:NewsArticle {url: $article.url})
    SET n.title = $article.title,
        n.published_at = $article.published_at,
        n.preview = $article.preview,
        n.content = $article.content,
        n.source_page = $article.source_page
    '''
    mention_query = '''
    MATCH (n:NewsArticle {url: $url})
    MERGE (p:Person {name: $person_name})
    MERGE (n)-[:MENTIONS]->(p)
    '''
    with driver.session(database=database) as session:
        for article in news:
            session.run(article_query, {'article': article})
            for person_name in detect_mentions(article, person_lookup):
                session.run(mention_query, {'url': article['url'], 'person_name': person_name})


In [7]:
driver = GraphDatabase.driver(
    require_env('NEO4J_URI'),
    auth=(require_env('NEO4J_USERNAME'), require_env('NEO4J_PASSWORD')),
)

try:
    create_constraints(driver, DATABASE)
    if CLEAR_SEMANTIC_GRAPH:
        clear_graph(driver, DATABASE)
    import_staff_semantic_entries(driver, DATABASE, semantic_entries)
    import_labs(driver, DATABASE, labs)
    person_lookup = build_person_lookup(semantic_entries, labs)
    import_news(driver, DATABASE, news, person_lookup)
finally:
    driver.close()

print('Graph import completed.')
print('Staff semantic entries:', len(semantic_entries))
print('Labs imported:', len(labs))
print('News articles imported:', len(news))


INFO neo4j.notifications: Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successful completion - index or constraint already exists. The command 'CREATE CONSTRAINT faculty_name IF NOT EXISTS FOR (e:Faculty) REQUIRE (e.name) IS UNIQUE' has no effect. The index or constraint specified by 'CONSTRAINT faculty_name FOR (e:Faculty) REQUIRE (e.name) IS UNIQUE' already exists.", position=None, raw_classification='SCHEMA', classification=<NotificationClassification.SCHEMA: 'SCHEMA'>, raw_severity='INFORMATION', severity=<NotificationSeverity.INFORMATION: 'INFORMATION'>, diagnostic_record={'_classification': 'SCHEMA', '_severity': 'INFORMATION', 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'CREATE CONSTRAINT faculty_name IF NOT EXISTS FOR (f:Faculty) REQUIRE f.name IS UNIQUE'
INFO neo4j.notifications: Received notification from DBMS server: <GqlStatusObject gql_status='00NA0', status_description="note: successf

Graph import completed.
Staff semantic entries: 97
Labs imported: 6
News articles imported: 8


## Step 5. Validate The Graph

These direct Cypher checks are helpful before moving to the query notebook.


In [8]:
def run_cypher(query: str, params: dict[str, Any] | None = None) -> list[dict[str, Any]]:
    driver = GraphDatabase.driver(
        require_env('NEO4J_URI'),
        auth=(require_env('NEO4J_USERNAME'), require_env('NEO4J_PASSWORD')),
    )
    try:
        with driver.session(database=DATABASE) as session:
            return [record.data() for record in session.run(query, params or {})]
    finally:
        driver.close()


In [9]:
run_cypher('''
MATCH (n)
RETURN labels(n) AS labels, count(*) AS total
ORDER BY total DESC
''')


[{'labels': ['Course'], 'total': 158},
 {'labels': ['Person'], 'total': 102},
 {'labels': ['Department'], 'total': 8},
 {'labels': ['NewsArticle'], 'total': 8},
 {'labels': ['Lab'], 'total': 6},
 {'labels': ['Faculty'], 'total': 1}]

In [10]:
run_cypher('''
MATCH (p:Person)-[:LEADS]->(l:Lab)
RETURN p.name AS person, l.name AS lab
ORDER BY lab
''')


[{'person': 'L. B. Pakholkiv',
  'lab': 'Educational Laboratory of Computer Modeling'},
 {'person': 'M. P. Onysko', 'lab': 'Information Systems'},
 {'person': 'V. B. Pasichnyk', 'lab': 'Information Technology'},
 {'person': 'L. B. Halamaha', 'lab': 'Programming'},
 {'person': 'O. B. Tomchii', 'lab': 'System Analysis'}]

In [11]:
run_cypher('''
MATCH (n:NewsArticle)-[:MENTIONS]->(p:Person)
RETURN n.title AS article, p.name AS person
ORDER BY article, person
LIMIT 25
''')


[{'article': 'Open Lecture by Associate Professor of the Department of Applied Mathematics Ihor Makar',
  'person': 'Ihor Makar'},
 {'article': 'We invite you to an open lecture by Assoc. Prof. Igor Makar',
  'person': 'Ihor Makar'}]